# Ch01-06: 범주형 인코딩 고급 기법 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

WoE/IV 구현

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

np.random.seed(42)
n = 2000
cats = np.random.choice(['Low','Mid','High','VHigh'], n, p=[0.4,0.3,0.2,0.1])
effects = {'Low':0.1,'Mid':0.3,'High':0.6,'VHigh':0.9}
y = np.array([int(np.random.random()<effects[c]) for c in cats])
df = pd.DataFrame({'cat':cats,'y':y})

def compute_woe_iv(df, feature, target):
    Np = (df[target]==1).sum(); Nn = (df[target]==0).sum()
    result = []
    for c in df[feature].unique():
        mask = df[feature]==c
        np_c = (df.loc[mask,target]==1).sum(); nn_c = (df.loc[mask,target]==0).sum()
        dp = np_c/Np if Np>0 else 0; dn = nn_c/Nn if Nn>0 else 0
        woe = np.log((dp+1e-10)/(dn+1e-10))
        iv_c = (dp-dn)*woe
        result.append({'category':c, 'n':mask.sum(), 'event_rate':np_c/mask.sum(), 'woe':woe, 'iv':iv_c})
    res = pd.DataFrame(result); total_iv = res['iv'].sum()
    return res, total_iv

r, iv = compute_woe_iv(df, 'cat', 'y')
print(r.round(4)); print(f"\nTotal IV: {iv:.4f}")

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(r['category'], r['woe'], color='steelblue')
axes[0].set_title('WoE by Category'); axes[0].set_ylabel('WoE')
axes[1].bar(r['category'], r['iv'], color='coral')
axes[1].set_title(f'IV by Category (Total={iv:.3f})')
plt.tight_layout(); plt.show()


**결과 해석**: IV가 0.1-0.3이면 보통 수준의 예측력. WoE는 각 범주의 타겟 관련 방향과 강도를 나타낸다.

---
## 문제 2 풀이

CatBoost Encoding 구현 + 과적합 비교

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold

np.random.seed(42)
n=1000; cats = np.random.choice([f'C{i}' for i in range(20)], n)
effects = {f'C{i}': 0.1+0.04*i for i in range(20)}
y = np.array([int(np.random.random()<effects[c]) for c in cats])

def catboost_enc(cats, y, a=1):
    p = y.mean(); encoded = np.zeros(len(cats))
    perm = np.random.permutation(len(cats))
    counts = {}; sums = {}
    for idx in perm:
        c = cats[idx]
        encoded[idx] = (sums.get(c,0)+a*p) / (counts.get(c,0)+a)
        counts[c] = counts.get(c,0)+1; sums[c] = sums.get(c,0)+y[idx]
    return encoded

def cv_target_enc(cats, y, n_folds=5):
    enc = np.full(len(cats), np.nan); gm = y.mean()
    kf = KFold(n_folds, shuffle=True, random_state=42)
    idx = np.arange(len(cats))
    for tr, val in kf.split(idx):
        means = {}
        for c in set(cats[tr]):
            mask = cats[tr]==c; means[c] = y[tr][mask].mean()
        enc[val] = [means.get(cats[v], gm) for v in val]
    return np.nan_to_num(enc, nan=gm)

# 비교
methods = {
    'Naive': pd.Series(cats).map(pd.DataFrame({'c':cats,'y':y}).groupby('c')['y'].mean()).values,
    'CV': cv_target_enc(cats, y),
    'CatBoost': catboost_enc(cats, y),
}

for name, X_enc in methods.items():
    train_score = LogisticRegression().fit(X_enc.reshape(-1,1), y).score(X_enc.reshape(-1,1), y)
    cv_score = cross_val_score(LogisticRegression(), X_enc.reshape(-1,1), y, cv=5).mean()
    print(f"{name:10s}: Train={train_score:.3f}, CV={cv_score:.3f}, Gap={train_score-cv_score:.3f}")


**결과 해석**: Naive는 train-CV gap이 크고 과적합. CV와 CatBoost는 정보 누출을 방지하여 gap이 작다.

---
## 문제 3 풀이

고카디널리티 인코딩 비교

In [ ]:
import numpy as np, pandas as pd, sys
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.feature_extraction import FeatureHasher

np.random.seed(42)
n = 5000; n_cats = 500
cats = np.array([f'cat_{np.random.randint(0,n_cats)}' for _ in range(n)])
effects = {f'cat_{i}': np.random.beta(2,5) for i in range(n_cats)}
y = np.array([int(np.random.random()<effects[c]) for c in cats])

# Frequency encoding
freq = pd.Series(cats).value_counts(normalize=True)
X_freq = np.array([freq[c] for c in cats]).reshape(-1,1)

# Target encoding (CV)
def cv_te(cats, y):
    enc = np.full(len(cats), y.mean())
    kf = KFold(5, shuffle=True, random_state=42)
    for tr, val in kf.split(np.arange(len(cats))):
        means = {}
        for c in set(cats[tr]): mask=cats[tr]==c; means[c]=y[tr][mask].mean()
        enc[val] = [means.get(cats[v], y.mean()) for v in val]
    return enc.reshape(-1,1)
X_te = cv_te(cats, y)

# Hash encoding
fh = FeatureHasher(n_features=64, input_type='string')
X_hash = fh.fit_transform([[c] for c in cats]).toarray()

for name, X_enc in [('Frequency',X_freq),('TargetCV',X_te),('Hash64',X_hash)]:
    cv = cross_val_score(LogisticRegression(max_iter=500), X_enc, y, cv=5).mean()
    mem = sys.getsizeof(X_enc) if isinstance(X_enc, np.ndarray) else X_enc.nbytes
    print(f"{name:12s}: CV AUC-proxy={cv:.3f}, Shape={X_enc.shape}, Mem~{mem//1024}KB")


**결과 해석**: Target Encoding(CV)이 가장 높은 예측 성능. Hash Encoding은 메모리 효율적이지만 충돌로 성능 저하 가능. Frequency는 간단하지만 타겟 정보 미반영.

---
## 문제 4 풀이

Target Encoding 편향-분산 분해

In [ ]:
import numpy as np, matplotlib.pyplot as plt

np.random.seed(42)
true_rates = {'A':0.3,'B':0.7,'C':0.5}
cat_sizes = {'A':200,'B':50,'C':10}
n_sims = 500

ms = [0,1,3,5,10,20,50,100]
bias2_all, var_all = {c:[] for c in true_rates}, {c:[] for c in true_rates}

for m in ms:
    for cat, true_p in true_rates.items():
        nc = cat_sizes[cat]; estimates = []
        for _ in range(n_sims):
            y_sim = np.random.binomial(1, true_p, nc)
            ybar = y_sim.mean(); gm = 0.5
            te = (nc*ybar + m*gm)/(nc+m)
            estimates.append(te)
        estimates = np.array(estimates)
        bias2_all[cat].append((estimates.mean()-true_p)**2)
        var_all[cat].append(estimates.var())

fig, axes = plt.subplots(1,3,figsize=(18,5))
for i, (cat, tp) in enumerate(true_rates.items()):
    nc = cat_sizes[cat]
    mse = np.array(bias2_all[cat])+np.array(var_all[cat])
    axes[i].plot(ms, bias2_all[cat], 'b-o', label='Bias²')
    axes[i].plot(ms, var_all[cat], 'r-s', label='Variance')
    axes[i].plot(ms, mse, 'g-^', label='MSE')
    axes[i].set_xlabel('m (smoothing)'); axes[i].set_title(f'{cat} (n={nc}, p={tp})')
    axes[i].legend(); axes[i].set_yscale('log')
plt.suptitle('Target Encoding: Bias-Variance Tradeoff'); plt.tight_layout(); plt.show()


**결과 해석**: m이 클수록 편향 증가+분산 감소. 소규모 범주(C, n=10)에서 m의 효과가 가장 크다. 최적 m은 범주 빈도에 따라 적응적으로 설정해야 한다.

---
## 확장 토론

### 인코딩 선택 가이드

| 상황 | 추천 |
|------|------|
| 저카디널리티 (<10) | One-Hot |
| 순서형 | Ordinal |
| 이진 타겟, 해석력 | WoE |
| 트리 모델 | Target/CatBoost |
| 초고카디널리티 | Hash + Target |